# 11 - Cell-Type Annotation & Deconvolution (Optional)

## Learning objectives
1. Annotate clusters/domains using marker-gene **module scores**.
2. Visualize cell-type-program scores spatially.
3. Understand what **deconvolution** is and which tools do it (and what they need).

## Concept
Because a spot is multi-cell, you often want to know **which cell types** contribute and in
what proportion. Without a matched single-cell reference we will **not fake** per-cell-type
fractions. Instead we do honest **marker module scoring** (average expression of a curated
gene set per spot) and then explain the proper deconvolution tools.


In [ ]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


In [ ]:
import scanpy as sc
import squidpy as sq

adata = st.load_adata('adata_clustered.h5ad')
adata


### Marker module scores
We define small, existence-checked marker sets for major mouse-brain programs and score
each spot with `sc.tl.score_genes` (mean of the set minus a random background).

In [ ]:
marker_sets = {
    'neuron':         ['Snap25', 'Rbfox3', 'Syt1', 'Meg3', 'Slc17a7'],
    'excitatory':     ['Slc17a7', 'Satb2', 'Neurod6'],
    'inhibitory':     ['Gad1', 'Gad2', 'Slc32a1'],
    'oligodendrocyte':['Mbp', 'Plp1', 'Mobp', 'Mog'],
    'astrocyte':      ['Gfap', 'Aqp4', 'Slc1a3'],
    'microglia':      ['Cx3cr1', 'C1qa', 'C1qb', 'Ctss'],
}
# For tumor datasets you would instead score: epithelial (EPCAM/KRT8),
# immune (PTPRC/CD3D), stromal (COL1A1/PECAM1), proliferation (MKI67).

scored = []
for name, genes in marker_sets.items():
    present = st.genes_present(adata, genes, verbose=False)
    if len(present) >= 2:  # need a couple of genes for a stable score
        sc.tl.score_genes(adata, present, score_name=f'score_{name}', random_state=st.SEED)
        scored.append(f'score_{name}')
print('Computed scores:', scored)


### Scores over the tissue

In [ ]:
sq.pl.spatial_scatter(adata, color=scored, ncols=3, size=1.3, cmap='viridis')


**Expected output:** program maps with clear anatomy - oligodendrocyte score on white-
matter tracts, neuronal score across gray matter, astrocyte/microglia more diffuse.

### Which program dominates each cluster?
Average each score within each Leiden cluster to give domains a tentative identity.

In [ ]:
import pandas as pd
if scored:
    by_cluster = adata.obs.groupby('clusters')[scored].mean()
    # Label each cluster by its highest-scoring program.
    by_cluster['dominant'] = by_cluster.idxmax(axis=1).str.replace('score_', '')
    display_cols = scored + ['dominant']
    by_cluster[display_cols]


### Proper deconvolution (when you have a reference)
To estimate **cell-type proportions per spot**, use a matched single-cell/nucleus RNA-seq
reference and a dedicated method. None are run here (they require that reference data and
heavier installs), but you should know the landscape:

| Tool | Approach | Needs |
|---|---|---|
| **cell2location** | Bayesian, estimates absolute abundances | scRNA reference + GPU helpful |
| **Tangram** | maps single cells onto spots (deep learning) | scRNA reference |
| **RCTD** (spacexr) | robust cell-type decomposition (R) | scRNA reference |
| **stereoscope** | probabilistic model | scRNA reference |
| **CARD** | spatially informed deconvolution (R) | scRNA reference |

Module scoring (above) answers *'how strong is program X here?'*; deconvolution answers
*'what fraction of this spot is cell type X?'* - a harder question needing a reference.

## Common pitfalls
- Treating a high module score as a cell-type *fraction* - it is a relative program
  strength, not a proportion.
- Using markers from the wrong species/tissue - always existence-check.
- Running deconvolution with a mismatched reference - garbage in, garbage out.

## Interpretation
Module scores give an honest, reference-free read on which cellular programs dominate each
domain, and we know exactly which tools to reach for when a reference is available.

## What this means biologically
Spots are mixtures; major brain cell programs (neuronal, oligodendrocyte, astrocyte,
microglial) localize sensibly, consistent with known anatomy. Quantifying *composition*
is the next rigor level and the basis for studying how cell-type mixtures shift in disease
or with treatment.

---
**Next:** `12_summary_research_extensions.ipynb`.
